# DATATHON 2026 HS Track - Task 1: MarketSense
## Multimodal Price Class Prediction for Rentals

**Goal:** predict the price class (`target`, 0–4) of a rental listing from tabular, text, and location signals.

**Approach (validated locally with 5-fold CV):**
1. The categorical columns `city` and `room_type` are *intentionally corrupted* (random casing, typos like `Entir3`/`Entyre`, separators, Indonesian variants). We normalize them with regex rules.
2. Only ~845 unique lat/lon pairs exist in 99k rows → location is a **cluster id**, the single strongest tabular feature (`latlon_key`).
3. ~70% of test `host_id`s appear in train → host-level aggregates are informative and leak-free (computed without the target).
4. **The nightly price is embedded in the free-text columns** of the competition data itself (`lattest comment`, `description`, `neighborhood_overview`, `name`, `host_about`) — e.g. `"Price info: 1237"`, `"only $95/night"`, `"RATE: FOURTEEN THOUSAND ..."`. Since `target` is a discretized price, extracting it (regex + number-word parser, ~99% coverage) is the strongest signal by far. **No external data is used** — everything is parsed from the provided train/test CSVs inside this notebook.
5. Model: **XGBoost** (hist, native categorical) with StratifiedKFold 5-fold; test prediction = average of fold probabilities.

| Version | Model | CV Acc | CV Macro-F1 | Notes |
|---|---|---|---|---|
| v1 | XGBoost, 94 features | 0.6173 | 0.5747 | tabular only |
| v2 | + TF-IDF/SVD text | 0.607 (fold 0) | - | text hurt → dropped |
| v5c | + 11 leaked-price features | **0.9016** | **0.8951** | this notebook |

## 1. Imports

In [19]:
import ast
import os
import re
import time

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold

## 2. Configuration

In [20]:
SEED = 42
N_FOLDS = 5
N_CLASS = 5

# Auto-detect Kaggle vs local execution.
# Returns the directory that DIRECTLY contains train.csv / test.csv.
def find_data_dir() -> str:
    kaggle_root = '/kaggle/input'
    if os.path.isdir(kaggle_root):
        for d, _, files in os.walk(kaggle_root):
            if 'train.csv' in files and 'test.csv' in files:
                return d
    # local: notebook lives in notebooks/, data in ../data/raw
    for cand in ('../data/raw', 'data/raw', '.'):
        if os.path.isfile(os.path.join(cand, 'train.csv')):
            return cand
    raise FileNotFoundError('train.csv not found — check data location')

DATA_DIR = find_data_dir()
print('data dir:', DATA_DIR)

XGB_PARAMS = dict(
    objective='multi:softprob', num_class=N_CLASS, eval_metric='mlogloss',
    learning_rate=0.05, max_depth=8, min_child_weight=5,
    subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
    tree_method='hist', seed=SEED, n_jobs=-1,
)

data dir: ../data/raw


## 3. Load Dataset

In [21]:
train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
y = train['target'].values

print('train:', train.shape, '| test:', test.shape)
train['target'].value_counts().sort_index()

train: (79200, 50) | test: (19800, 49)


target
0     4759
1    16635
2    30885
3    21377
4     5544
Name: count, dtype: int64

## 4. Exploratory Data Analysis

Key findings that drive every later decision:
- **Target imbalance:** class 2 ≈ 39%, classes 0 and 4 ≈ 6–7% → use *Stratified*KFold and report macro-F1 next to accuracy.
- **`city` / `room_type` are noisy on purpose** (`CITY-C`, `cty_c`, `ID_City_C_01`, `Entir3 home`, `apartemen`, `pRiVaTe RoOm`, …).
- **`calendar_updated` is 100% missing** → dropped.
- **~845 unique coordinates** for 99,000 rows → listings sit on a grid of location clusters.
- Higher price classes (3–4) have *fewer reviews* and *lower estimated occupancy* expensive places get booked less.

In [22]:
print('missing % (top 12):')
print((train.isna().mean() * 100).round(1).sort_values(ascending=False).head(12))
print()
print('unique lat/lon pairs:', train.groupby(['latitude', 'longitude']).ngroups)
print('noisy city examples:', train['city'].sample(5, random_state=SEED).tolist())
print()
print('mean occupancy / reviews per class:')
print(train.groupby('target')[['estimated_occupancy_l365d', 'number_of_reviews']].mean().round(1))

missing % (top 12):
calendar_updated               100.0
license                         63.3
neighborhood_overview           57.9
neighbourhood                   57.9
host_about                      42.4
host_location                   24.3
first_review                    19.8
last_review                     19.8
review_scores_rating            19.8
review_scores_accuracy          19.8
review_scores_communication     19.8
review_scores_cleanliness       19.8
dtype: float64

unique lat/lon pairs: 845
noisy city examples: ['city_c', '  City_B ', 'City_A_M3tropolitan', 'city_b', 'City_A!']

mean occupancy / reviews per class:
        estimated_occupancy_l365d  number_of_reviews
target                                              
0                            84.9               40.9
1                            85.7               40.3
2                            77.1               38.8
3                            58.8               32.4
4                            30.2               18

## 5. Data Cleaning

Regex normalizers for the corrupted categoricals. Leet-speak (`3`→`e`) is undone first, then keyword matching maps every variant to a canonical value. Verified on train: 0 rows end up in an unmapped bucket for `room_type`; every `city` string contains the city letter and is recovered by the pattern.

In [23]:
def normalize_city(s: str) -> str:
    """'CITY-C', 'cty_c', 'ID_City_B_01', 'City_C_M3tropolitan' -> 'c', 'b', ..."""
    s = str(s).lower().replace('3', 'e')
    m = re.search(r'(?:city|cty)[^a-z]*([abcd])(?![a-z])', s)
    return m.group(1) if m else 'unk'


def normalize_room_type(s: str) -> str:
    s = str(s).lower().replace('3', 'e')
    s = re.sub(r'[^a-z]', ' ', s)
    if any(k in s for k in ('entire', 'entyre', 'entir', 'whole', 'full house',
                            'apartemen', 'residential')):
        return 'entire'
    if any(k in s for k in ('privat', 'privte', 'own room')):
        return 'private'
    if any(k in s for k in ('shar', 'shre', 'kamar')):
        return 'shared'
    if any(k in s for k in ('hotel', 'hotell', 'hoteel', 'htl', 'boutique')):
        return 'hotel'
    return 'other'


def parse_percent(s) -> float:
    if pd.isna(s):
        return np.nan
    m = re.search(r'(\d+)', str(s))
    return float(m.group(1)) if m else np.nan


def parse_bathrooms_text(s):
    """'1.5 shared baths' -> (1.5, shared=1, private=0). 'Half-bath' -> 0.5."""
    if pd.isna(s):
        return np.nan, 0, 0
    s = str(s).lower()
    shared = int('shared' in s)
    private = int('private' in s)
    m = re.search(r'(\d+(?:\.\d+)?)', s)
    n = float(m.group(1)) if m else (0.5 if 'half' in s else np.nan)
    return n, shared, private


# sanity check on train
print(train['city'].map(normalize_city).value_counts().to_dict())
print(train['room_type'].map(normalize_room_type).value_counts().to_dict())

{'c': 30936, 'b': 20384, 'a': 18618, 'd': 9262}
{'entire': 64562, 'private': 13606, 'shared': 802, 'hotel': 230}


### 5b. Leaked-Price Extraction

The strongest signal in this competition: **the listing's nightly price is written inside the free-text columns** (planted by the organizers as a hidden signal). Examples found in the data:

- `"Price info: 1237"`, `"rate: 95!"`, `"only $2,450 per night"`, `"it costs 13.377"`
- spelled-out: `"RATE: FOURTEEN THOUSAND TWO HUNDRED AND SIX!"`

Since `target` is a discretized price, recovering this number ≈ recovering the label. The extractor below:

1. tries 14 numeric regex patterns (price labels, `$N/night`, `only N!`, `k` suffix, thousand-separator disambiguation),
2. falls back to a **number-word parser** for spelled-out amounts (regex alternation lists longer words first, so `nineteen` matches before `nine`),
3. scans the 5 text columns in priority order and returns the first hit.

Coverage: **~99%** of all rows. Prices are on different scales per city (different currencies), which is handled later with city-relative features. This is *not* target leakage in the modeling sense (no `target` used, identical transform for train and test) and uses only the competition CSVs.

In [24]:
# ---------- number-word parser ----------
UNITS = {
    'zero': 0, 'one': 1, 'two': 2, 'three': 3, 'four': 4, 'five': 5,
    'six': 6, 'seven': 7, 'eight': 8, 'nine': 9, 'ten': 10, 'eleven': 11,
    'twelve': 12, 'thirteen': 13, 'fourteen': 14, 'fifteen': 15,
    'sixteen': 16, 'seventeen': 17, 'eighteen': 18, 'nineteen': 19,
}
TENS = {'twenty': 20, 'thirty': 30, 'forty': 40, 'fifty': 50,
        'sixty': 60, 'seventy': 70, 'eighty': 80, 'ninety': 90}
SCALES = {'hundred': 100, 'thousand': 1000, 'million': 1000000}
# longest alternatives first: regex alternation is first-match, so 'nineteen'
# must precede 'nine' or "nineteen thousand" parses as just "nine".
NUMWORD = r'(?:seventeen|thirteen|fourteen|eighteen|nineteen|seventy|sixteen|fifteen|hundred|thousand|million|eleven|twelve|twenty|thirty|eighty|ninety|forty|fifty|sixty|three|seven|eight|zero|four|five|nine|one|two|six|ten|and|[-\s,])+'


def words_to_num(text: str) -> float:
    tokens = re.split(r'[\s,-]+', text.lower())
    total, current = 0, 0
    seen = False
    for tok in tokens:
        if tok in UNITS:
            current += UNITS[tok]; seen = True
        elif tok in TENS:
            current += TENS[tok]; seen = True
        elif tok == 'hundred':
            current = max(current, 1) * 100; seen = True
        elif tok in ('thousand', 'million'):
            total += max(current, 1) * SCALES[tok]; current = 0; seen = True
        elif tok in ('and', ''):
            continue
        else:
            break
    return float(total + current) if seen else np.nan


NUMG = r'\$?\s*([\d.,]+)\s*([kK])?'
PATTERNS = [
    # explicit price/rate labels
    rf'price\s*info\s*:?\s*{NUMG}',
    rf'(?:price|rate)\s*(?:is)?\s*(?:only)?\s*:\s*{NUMG}',
    rf'(?:the\s+)?(?:price|rate)\s+is\s+only\s+{NUMG}',
    rf'special\s+rate\s+{NUMG}\s+for\s+this\s+week',
    rf'worth\s+it,?\s+only\s+{NUMG}\s*per\s*night',
    rf'value\s+for\s+money\s+at\s+{NUMG}\s*(?:per\s*night)?',
    rf'{NUMG}\s+is\s+the\s+price',
    rf'it\s+costs?\s+{NUMG}',
    rf'(?:price|rate)\s*:?\s*{NUMG}\s*!',
    rf'only\s+\$\s*([\d.,]+)\s*([kK])?\s*!',
    rf'worth\s+the\s+price\s+only\s+{NUMG}\s*!?',
    rf'only\s+([\d.,]+)\s*([kK])?\s*!',
    rf'only\s+{NUMG}\s*per\s*night',
    rf'\$\s*([\d.,]+)\s*([kK])?\s*(?:per|/|a)\s*night',
]
PATTERNS = [re.compile(p, re.I) for p in PATTERNS]

WORD_PATTERNS = [
    re.compile(rf'(?:price|rate)\s*:?\s*({NUMWORD})\s*!', re.I),
    re.compile(rf'(?:price|rate)\s*:\s*({NUMWORD})', re.I),
    re.compile(rf'(?:the\s+)?(?:price|rate)\s+is\s+only\s+({NUMWORD})', re.I),
    re.compile(rf'({NUMWORD})\s+is\s+the\s+(?:price|rate)', re.I),
    re.compile(rf'it\s+costs?\s+({NUMWORD})', re.I),
    re.compile(rf'special\s+rate\s+({NUMWORD})\s+for', re.I),
    re.compile(rf'worth\s+it,?\s+only\s+({NUMWORD})\s+per\s+night', re.I),
    re.compile(rf'value\s+for\s+money\s+at\s+({NUMWORD})\s+per\s+night', re.I),
]


def parse_num(numstr: str, ksuf) -> float:
    s = numstr.strip().rstrip('.').rstrip(',')
    has_k = bool(ksuf)
    try:
        if ',' in s and '.' in s:
            val = float(s.replace(',', ''))
        elif ',' in s:
            parts = s.split(',')
            val = float(s.replace(',', '')) if all(len(p) == 3 for p in parts[1:]) \
                else float(s.replace(',', '.'))
        elif '.' in s:
            parts = s.split('.')
            if len(parts) == 2 and len(parts[-1]) == 2:
                val = float(s)                      # 173.00
            elif all(len(p) == 3 for p in parts[1:]) and not has_k:
                val = float(s.replace('.', ''))     # 13.377 -> 13377
            else:
                val = float(s)                      # 0.327 (with k) / 3.189 (amb.)
        else:
            val = float(s)
    except ValueError:
        return np.nan
    if has_k:
        val *= 1000.0
    # '3.189' with no k and 3 decimals already handled as 3189
    return val


def extract_price_from_text(txt: str) -> float:
    for pat in PATTERNS:
        m = pat.search(txt)
        if m:
            v = parse_num(m.group(1), m.group(2) if (m.lastindex or 0) >= 2 else None)
            if v and v > 0:
                return v
    for pat in WORD_PATTERNS:
        m = pat.search(txt)
        if m:
            v = words_to_num(m.group(1))
            if v and v > 0:
                return v
    return np.nan


def extract_price(row) -> float:
    for col in ('lattest comment', 'description', 'neighborhood_overview', 'name',
                'host_about'):
        txt = row.get(col)
        if pd.isna(txt):
            continue
        v = extract_price_from_text(str(txt))
        if not np.isnan(v):
            return v
    return np.nan


t0 = time.time()
train['leak_price'] = train.apply(extract_price, axis=1)
test['leak_price'] = test.apply(extract_price, axis=1)
print(f'extraction {time.time()-t0:.0f}s')
print(f'coverage train: {train["leak_price"].notna().mean():.4f} '
      f'| test: {test["leak_price"].notna().mean():.4f}')
print('\nmedian extracted price per target class (train):')
print(train.dropna(subset=['leak_price']).groupby('target')['leak_price']
      .agg(['count', 'median']).round(0))

extraction 42s
coverage train: 0.9904 | test: 0.9907

median extracted price per target class (train):
        count  median
target               
0        4707   588.0
1       16459   794.0
2       30599  1221.0
3       21166  2086.0
4        5510  8202.0


## 6. Feature Engineering

Built on train+test combined (**no target used** → no leakage):
- normalized categoricals + `latlon_key` (rounded coordinate pair = location cluster)
- numeric passthrough, percent parsing, boolean flags, `has_license`
- date deltas relative to `date_obtained` (host tenure, review recency)
- amenities: count + 24 keyword flags (pool, hot tub, dishwasher… premium markers)
- text *lengths* only (word counts); TF-IDF/SVD was tested and hurt CV
- ratios (beds per person, availability ratio) and host / location aggregates
- **11 leaked-price features** (v5c): raw / log price, per-person / per-bed / per-bedroom, percentile rank within city / location cluster / neighbourhood, log-price relative to cluster & city medians, and a `has_leak` flag. The *rank-within-city* features are the key trick — prices are on different currency scales per city, so the city-relative rank is what maps to the global price class.

In [25]:
KEY_AMENITIES = [
    'wifi', 'kitchen', 'air conditioning', 'pool', 'free parking', 'gym',
    'elevator', 'washer', 'dryer', 'dishwasher', 'bathtub', 'balcony',
    'heating', 'tv', 'coffee', 'hot tub', 'bbq', 'crib', 'workspace',
    'self check-in', 'lockbox', 'breakfast', 'long term stays', 'pets allowed',
]


def count_words(s) -> int:
    return 0 if pd.isna(s) else len(str(s).split())


def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """Pure per-row transforms (no target usage)."""
    out = pd.DataFrame(index=df.index)

    # normalized categoricals
    out['city'] = df['city'].map(normalize_city)
    out['room_type'] = df['room_type'].map(normalize_room_type)
    out['property_type'] = (df['property_type'].astype(str).str.lower()
                            .str.replace(r'[^a-z ]', ' ', regex=True)
                            .str.replace(r'\s+', ' ', regex=True).str.strip())
    out['neighbourhood'] = df['neighbourhood'].fillna('missing')
    out['host_response_time'] = df['host_response_time'].fillna('missing')

    # numeric passthrough
    num_cols = ['accommodates', 'bathrooms', 'bedrooms', 'beds',
                'minimum_nights', 'maximum_nights',
                'availability_30', 'availability_60', 'availability_90',
                'availability_365', 'number_of_reviews',
                'estimated_occupancy_l365d', 'review_scores_rating',
                'review_scores_accuracy', 'review_scores_cleanliness',
                'review_scores_communication', 'reviews_per_month',
                'host_total_listings_count', 'latitude', 'longitude']
    for c in num_cols:
        out[c] = pd.to_numeric(df[c], errors='coerce')

    # percentages
    out['host_response_rate'] = df['host_response_rate'].map(parse_percent)
    out['host_acceptance_rate'] = df['host_acceptance_rate'].map(parse_percent)

    # booleans
    for c in ['host_has_profile_pic', 'host_identity_verified',
              'has_availability', 'instant_bookable']:
        out[c] = (df[c] == 't').astype(int)
    out['has_license'] = df['license'].notna().astype(int)

    # bathrooms_text
    bt = df['bathrooms_text'].map(parse_bathrooms_text)
    out['bath_n'] = [t[0] for t in bt]
    out['bath_shared'] = [t[1] for t in bt]
    out['bath_private'] = [t[2] for t in bt]
    out['bathrooms'] = out['bathrooms'].fillna(out['bath_n'])

    # date deltas
    ref = pd.to_datetime(df['date_obtained'], errors='coerce')
    for c in ['host_since', 'first_review', 'last_review']:
        d = pd.to_datetime(df[c], errors='coerce')
        out[f'days_{c}'] = (ref - d).dt.days

    # amenities
    def parse_amen(s):
        try:
            return [a.lower() for a in ast.literal_eval(str(s))]
        except (ValueError, SyntaxError):
            return []
    amen = df['amenities'].map(parse_amen)
    out['n_amenities'] = amen.map(len)
    for k in KEY_AMENITIES:
        out[f'am_{k.replace(" ", "_")}'] = amen.map(
            lambda lst, k=k: int(any(k in a for a in lst)))

    # host_verifications
    hv = df['host_verifications'].fillna('[]').astype(str)
    out['n_verifications'] = hv.str.count(',') + hv.str.contains(r'\w').astype(int)
    out['verif_email'] = hv.str.contains('email').astype(int)
    out['verif_phone'] = hv.str.contains('phone').astype(int)
    out['verif_work_email'] = hv.str.contains('work_email').astype(int)

    # text lengths (word counts only — raw TF-IDF hurt CV)
    for c in ['name', 'description', 'neighborhood_overview', 'host_about',
              'lattest comment']:
        out[f'len_{c.replace(" ", "_")}'] = df[c].map(count_words)

    # ratios / interactions
    out['beds_per_person'] = out['beds'] / out['accommodates'].clip(lower=1)
    out['baths_per_person'] = out['bathrooms'] / out['accommodates'].clip(lower=1)
    out['beds_per_bedroom'] = out['beds'] / out['bedrooms'].clip(lower=1)
    out['occ_x_reviews'] = (out['estimated_occupancy_l365d']
                            * np.log1p(out['number_of_reviews']))
    out['avail_ratio_30_365'] = out['availability_30'] / (out['availability_365'] + 1)
    out['review_span_days'] = out['days_first_review'] - out['days_last_review']
    out['reviews_per_day_active'] = (out['number_of_reviews']
                                     / out['review_span_days'].clip(lower=1))
    out['min_nights_log'] = np.log1p(out['minimum_nights'])
    out['max_nights_log'] = np.log1p(out['maximum_nights'].clip(upper=10000))

    # location cluster id (~845 unique coordinates in the whole dataset)
    out['latlon_key'] = (out['latitude'].round(3).astype(str) + '_'
                         + out['longitude'].round(3).astype(str))
    return out


def add_group_features(all_df: pd.DataFrame, raw_all: pd.DataFrame) -> pd.DataFrame:
    """Aggregates over train+test combined (no target involved -> no leakage)."""
    out = all_df.copy()

    host = raw_all.groupby('host_id').agg(
        host_n_listings=('id', 'count'),
        host_mean_reviews=('number_of_reviews', 'mean'),
        host_mean_occ=('estimated_occupancy_l365d', 'mean'),
        host_mean_accom=('accommodates', 'mean'),
    )
    out = out.join(host, on=raw_all['host_id'])
    out['host_id_freq'] = out['host_n_listings']

    grp = out.groupby('latlon_key')
    out['loc_count'] = grp['accommodates'].transform('count')
    out['loc_mean_accom'] = grp['accommodates'].transform('mean')
    out['loc_mean_occ'] = grp['estimated_occupancy_l365d'].transform('mean')
    out['loc_mean_reviews'] = grp['number_of_reviews'].transform('mean')

    out['neigh_freq'] = out.groupby('neighbourhood')['accommodates'].transform('count')
    out['prop_freq'] = out.groupby('property_type')['accommodates'].transform('count')
    return out


def add_leak_features(all_df: pd.DataFrame, raw_all: pd.DataFrame) -> pd.DataFrame:
    """11 features derived from the extracted leaked price (no target involved).

    Prices are on different scales per city (different currencies), so the
    percentile rank WITHIN city / location cluster is the feature that maps
    cleanly to the global price class."""
    out = all_df.copy()
    lp = raw_all['leak_price']
    out['leak_price'] = lp
    out['leak_log'] = np.log1p(lp)
    out['leak_per_person'] = lp / out['accommodates'].clip(lower=1)
    out['leak_per_bed'] = lp / out['beds'].clip(lower=1)
    out['leak_per_bedroom'] = lp / out['bedrooms'].clip(lower=1)
    # percentile within city / cluster / neighbourhood (train+test combined, no target)
    out['leak_pct_city'] = out.groupby('city')['leak_log'].rank(pct=True)
    out['leak_pct_loc'] = out.groupby('latlon_key')['leak_log'].rank(pct=True)
    out['leak_pct_neigh'] = out.groupby('neighbourhood')['leak_log'].rank(pct=True)
    # relative to cluster / city median
    out['leak_rel_loc'] = out['leak_log'] - out.groupby('latlon_key')['leak_log'].transform('median')
    out['leak_rel_city'] = out['leak_log'] - out.groupby('city')['leak_log'].transform('median')
    out['has_leak'] = lp.notna().astype(int)
    return out


t0 = time.time()
raw_all = pd.concat([train.drop(columns=['target']), test], ignore_index=True)
feats = build_features(raw_all)
feats = add_group_features(feats, raw_all)
feats = add_leak_features(feats, raw_all)

CAT_COLS = ['city', 'room_type', 'property_type', 'neighbourhood',
            'host_response_time', 'latlon_key']
for c in CAT_COLS:
    feats[c] = feats[c].astype('category')

X = feats.iloc[:len(train)].reset_index(drop=True)
X_test = feats.iloc[len(train):].reset_index(drop=True)
print(f'{X.shape[1]} features, prep {time.time()-t0:.0f}s')

105 features, prep 31s


## 7. Train / Validation Split

**StratifiedKFold (5 folds)** preserves the 6/21/39/27/7% class mix in every fold. Every row is used for validation exactly once (out-of-fold), so the CV score is an unbiased estimate and no data is wasted.

In [26]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(skf.split(X, y))
for i, (itr, iva) in enumerate(folds):
    print(f'fold {i}: train={len(itr)} valid={len(iva)}')

fold 0: train=63360 valid=15840
fold 1: train=63360 valid=15840
fold 2: train=63360 valid=15840
fold 3: train=63360 valid=15840
fold 4: train=63360 valid=15840


## 8. Model Training

XGBoost `hist` with native categorical support. Early stopping on each fold's validation set (100 rounds patience) picks the tree count automatically no manual tuning of `n_estimators`.

In [27]:
t0 = time.time()
oof = np.zeros((len(X), N_CLASS))
pred = np.zeros((len(X_test), N_CLASS))
dtest = xgb.DMatrix(X_test, enable_categorical=True)

for fold, (itr, iva) in enumerate(folds):
    dtr = xgb.DMatrix(X.iloc[itr], y[itr], enable_categorical=True)
    dva = xgb.DMatrix(X.iloc[iva], y[iva], enable_categorical=True)
    model = xgb.train(XGB_PARAMS, dtr, num_boost_round=4000,
                      evals=[(dva, 'va')], early_stopping_rounds=100,
                      verbose_eval=False)
    best = model.best_iteration + 1
    oof[iva] = model.predict(dva, iteration_range=(0, best))
    pred += model.predict(dtest, iteration_range=(0, best)) / N_FOLDS
    print(f'fold {fold}: acc={accuracy_score(y[iva], oof[iva].argmax(1)):.4f} '
          f'best_iter={model.best_iteration} ({time.time()-t0:.0f}s)')

fold 0: acc=0.9018 best_iter=491 (107s)
fold 1: acc=0.9028 best_iter=497 (191s)
fold 2: acc=0.9000 best_iter=461 (268s)
fold 3: acc=0.9037 best_iter=525 (355s)
fold 4: acc=0.8997 best_iter=397 (427s)


## 9. Cross Validation

Out-of-fold metrics, this is the number we trust, *not* the public LB (only ~50% of test).

In [28]:
oof_lbl = oof.argmax(1)
print(f'OOF accuracy   : {accuracy_score(y, oof_lbl):.4f}')
print(f'OOF macro-F1   : {f1_score(y, oof_lbl, average="macro"):.4f}')
print(f'OOF weighted-F1: {f1_score(y, oof_lbl, average="weighted"):.4f}')
print()
has = X['has_leak'].values == 1
print(f'acc rows WITH leak ({has.mean():.2%}): {accuracy_score(y[has], oof_lbl[has]):.4f}')
print(f'acc rows WITHOUT leak: {accuracy_score(y[~has], oof_lbl[~has]):.4f}')
print()
print('confusion matrix (rows = true class):')
print(confusion_matrix(y, oof_lbl))
print()
imp = pd.Series(model.get_score(importance_type='gain')).sort_values(ascending=False)
print('top 15 features by gain:')
print(imp.head(15).round(1))

OOF accuracy   : 0.9016
OOF macro-F1   : 0.8950
OOF weighted-F1: 0.9016

acc rows WITH leak (99.04%): 0.9056
acc rows WITHOUT leak: 0.4875

confusion matrix (rows = true class):
[[ 4096   560    66    35     2]
 [  408 14658  1371   148    50]
 [   27  1291 28121  1267   179]
 [   38    30  1356 19476   477]
 [   23    14    32   418  5057]]

top 15 features by gain:
leak_pct_city       36.2
leak_rel_city       26.7
accommodates        24.3
has_license         21.9
city                19.3
leak_per_person     17.0
room_type           11.5
baths_per_person     7.3
leak_price           6.8
latitude             6.0
latlon_key           5.5
longitude            5.4
property_type        4.8
leak_log             4.7
leak_rel_loc         4.5
dtype: float64


## 10. Prediction

Test prediction = mean of the 5 fold models' probabilities → argmax. Averaging probabilities is a free mini-ensemble: it reduces variance versus training one model on all data.

In [29]:
test_lbl = pred.argmax(1)
print('predicted class distribution (test):')
print(pd.Series(test_lbl).value_counts(normalize=True).sort_index().round(3))
print('train distribution for comparison:')
print(pd.Series(y).value_counts(normalize=True).sort_index().round(3))

predicted class distribution (test):
0    0.059
1    0.209
2    0.390
3    0.269
4    0.073
Name: proportion, dtype: float64
train distribution for comparison:
0    0.06
1    0.21
2    0.39
3    0.27
4    0.07
Name: proportion, dtype: float64


## 11. Submission

In [30]:
submission = pd.DataFrame({'id': test['id'], 'target': test_lbl})
submission.to_csv('submission.csv', index=False)
print(submission.shape)
submission.head()

(19800, 2)


,id,target
0,16838,3
1,43583,3
2,79935,2
3,54219,2
4,51611,2
